# Runtime Agents - Core Concepts (Code-Focused)

A hands-on guide focusing on **code examples** rather than lengthy explanations.

**What you'll learn:** The core components through executable code.

---

## Setup

In [ ]:
import os
import sys
from pathlib import Path

# Add project to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Load environment
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print(f"✓ Project: {project_root}")
print(f"✓ API Key: {'Loaded' if os.getenv('OPENAI_API_KEY') else 'Missing'}")

---
## 1. LLM Client - Talk to OpenAI

In [ ]:
from runtime_agents.shared.llm import OpenAIChatClient, Message

# Create client
llm = OpenAIChatClient(
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-4o-mini",
    base_url="https://api.openai.com"
)

# Simple chat
messages = [
    Message("system", "You are a helpful assistant."),
    Message("user", "Explain agents in one sentence.")
]

response = await llm.chat(messages, temperature=0.7)
print(response)

---
## 2. Tools - Agent Capabilities

In [ ]:
from runtime_agents.shared.tools import TimeTool, HttpGetTool

# Tool 1: Get current time
time_tool = TimeTool()
result = await time_tool({})
print(f"Time: {result}")

# Tool 2: Fetch URL
http_tool = HttpGetTool()
result = await http_tool({"url": "https://example.com"})
print(f"\nHTTP Status: {result['status']}")
print(f"Content: {result['text'][:100]}...")

In [ ]:
# Get all default tools
from utils.tool_registry import get_default_tools

tools = get_default_tools()
print("Available tools:")
for name in tools.keys():
    print(f"  - {name}")

---
## 3. Agent Templates - Define Agent Roles

In [ ]:
from runtime_agents.template_based.agents import AgentTemplate

# Define a template
analyst = AgentTemplate(
    key="analyst",
    name="Data Analyst",
    system_prompt="You analyze data and provide insights.",
    tool_names=["file_read", "time_now"]
)

print(f"Template: {analyst.name}")
print(f"Tools: {analyst.tool_names}")

---
## 4. Agent Instance - Spawn and Run

In [ ]:
from runtime_agents.template_based.agents import AgentInstance

# Create template
helper = AgentTemplate(
    key="helper",
    name="Helper",
    system_prompt="You help users with quick questions.",
    tool_names=["time_now"]
)

# Spawn instance with scoped tools
scoped_tools = {"time_now": tools["time_now"]}
agent = AgentInstance(template=helper, llm=llm, tools=scoped_tools)

# Run it
result = await agent.run("What's the current time?")
print(f"Agent: {result.agent_name}")
print(f"Output: {result.output}")

---
## 5. Orchestrator - Coordinate Multiple Agents

In [ ]:
from runtime_agents.template_based.orchestrator import Orchestrator

# Create agent registry
registry = {
    "planner": AgentTemplate(
        key="planner",
        name="Planner",
        system_prompt="You break down requests into plans.",
        tool_names=["time_now"]
    ),
    "writer": AgentTemplate(
        key="writer",
        name="Writer",
        system_prompt="You write clear, concise outputs.",
        tool_names=["time_now"]
    )
}

# Create orchestrator
orch = Orchestrator(llm=llm, registry=registry, tools=tools, session_context="")

# Run a request
agent_results, final = await orch.run("Explain what Python is.")

print(f"Agents used: {len(agent_results)}")
for r in agent_results:
    print(f"  - {r.agent_name}")
print(f"\nFinal answer: {final[:200]}...")

---
## 6. Session Management - Persist State

In [ ]:
from utils.session_manager import SessionManager

# Create manager
sm = SessionManager()

# Create session
session = sm.create_session("demo_123")
session.add_message("user", "Hello!")
session.add_message("assistant", "Hi there!")

# Save
sm.save_session(session)

# Load
loaded = sm.load_session("demo_123")
print(f"Session: {loaded.session_id}")
print(f"Messages: {len(loaded.chat_history)}")
for msg in loaded.chat_history:
    print(f"  {msg['role']}: {msg['content']}")

---
## 7. Agent Factory - Switch Architectures

In [ ]:
from utils.agent_factory import AgentFactory

factory = AgentFactory()

# Create template-based orchestrator
orch1 = factory.create_orchestrator(
    llm=llm,
    tools=tools,
    session_context="",
    agent_type="template_based"
)

# Create meta-agent orchestrator
orch2 = factory.create_orchestrator(
    llm=llm,
    tools=tools,
    session_context="",
    agent_type="meta"
)

print(f"Orchestrator 1: {type(orch1).__name__}")
print(f"Orchestrator 2: {type(orch2).__name__}")

---
## 8. Complete Example - End-to-End

In [ ]:
# Full workflow: Session → Factory → Orchestrator → Agents → Result

# 1. Create session
session = sm.create_session("e2e_demo")
session.add_message("user", "Explain async programming in Python.")

# 2. Create orchestrator via factory
orch = factory.create_orchestrator(
    llm=llm,
    tools=tools,
    session_context="",
    agent_type="template_based"
)

# 3. Run
agent_results, final = await orch.run("Explain async programming in Python.")

# 4. Save to session
session.add_message("assistant", final)
sm.save_session(session)

# 5. Display
print(f"Agents: {[r.agent_name for r in agent_results]}")
print(f"\nAnswer: {final}")

---
## 9. Custom Agent - Build Your Own

In [ ]:
# Define custom template
code_reviewer = AgentTemplate(
    key="code_reviewer",
    name="Code Reviewer",
    system_prompt="You review code for quality and suggest improvements.",
    tool_names=["time_now"]
)

# Spawn and run
reviewer = AgentInstance(
    template=code_reviewer,
    llm=llm,
    tools={"time_now": tools["time_now"]}
)

code = '''
def add(a, b):
    return a + b
'''

result = await reviewer.run(f"Review this code:\n{code}")
print(result.output)

---
## 10. Custom Tool - Build Your Own

In [ ]:
from dataclasses import dataclass
from typing import Dict, Any

@dataclass
class CalculatorTool:
    name: str = "calculator"
    description: str = "Perform basic math. Input: {operation: 'add'|'multiply', a: number, b: number}"
    
    async def __call__(self, input: Dict[str, Any]) -> Dict[str, Any]:
        op = input.get("operation")
        a = input.get("a", 0)
        b = input.get("b", 0)
        
        if op == "add":
            return {"result": a + b}
        elif op == "multiply":
            return {"result": a * b}
        else:
            return {"error": "Unknown operation"}

# Test it
calc = CalculatorTool()
result = await calc({"operation": "add", "a": 5, "b": 3})
print(f"5 + 3 = {result['result']}")

result = await calc({"operation": "multiply", "a": 4, "b": 7})
print(f"4 × 7 = {result['result']}")

---
## 11. Alternative Architectures

In [ ]:
# LLM-Generated: Agents created dynamically by LLM
from runtime_agents_llm_generated.orchestrator import LLMGeneratedOrchestrator

llm_gen = LLMGeneratedOrchestrator(
    llm=llm,
    available_tools=tools,
    session_context="",
    max_agents=2
)

results, final = await llm_gen.run("Explain machine learning.")
print(f"LLM-Generated: {len(results)} agents")
print(f"Answer: {final[:150]}...")

In [ ]:
# Meta-Agent: Single adaptive agent
from runtime_agents_meta.orchestrator import MetaOrchestrator

meta = MetaOrchestrator(
    llm=llm,
    available_tools=tools,
    session_context=""
)

results, final = await meta.run("What is recursion?")
print(f"Meta-Agent: {len(results)} agents")
print(f"Answer: {final[:150]}...")

---
## 12. Performance Tracking

In [ ]:
from utils.performance_tracker import PerformanceTracker
import time

tracker = PerformanceTracker()

# Record execution
tracker.record_execution(
    agent_type="template_based",
    execution_time=2.5,
    token_usage={"input_tokens": 500, "output_tokens": 300},
    cost_estimate=0.002,
    num_agents_spawned=2,
    tool_calls_count=1
)

tracker.save_metrics()

# Get stats
stats = tracker.get_comparison_stats()
print("Performance stats:")
for arch, metrics in stats.items():
    print(f"  {arch}: {metrics}")

---
## 13. Architecture Comparison

In [ ]:
# Compare different architectures on same query
query = "What is Python?"

architectures = ["template_based", "meta"]
results = {}

for arch in architectures:
    orch = factory.create_orchestrator(llm, tools, "", agent_type=arch)
    start = time.time()
    agent_results, final = await orch.run(query)
    duration = time.time() - start
    
    results[arch] = {
        "agents": len(agent_results),
        "time": f"{duration:.2f}s",
        "answer_length": len(final)
    }

print("Comparison:")
for arch, data in results.items():
    print(f"  {arch}: {data}")

---
## 14. Context Passing

In [ ]:
# Demonstrate how context accumulates

# Agent 1: Researcher
researcher = AgentInstance(
    template=AgentTemplate(
        key="researcher",
        name="Researcher",
        system_prompt="You gather facts.",
        tool_names=["time_now"]
    ),
    llm=llm,
    tools={"time_now": tools["time_now"]}
)

result1 = await researcher.run("What is Python?", context=None)
print(f"Researcher: {result1.output[:100]}...")

# Agent 2: Writer (receives context from researcher)
writer = AgentInstance(
    template=AgentTemplate(
        key="writer",
        name="Writer",
        system_prompt="You write summaries.",
        tool_names=["time_now"]
    ),
    llm=llm,
    tools={"time_now": tools["time_now"]}
)

context = f"[Researcher]\n{result1.output}"
result2 = await writer.run("Summarize the above.", context=context)
print(f"\nWriter (with context): {result2.output[:100]}...")

---
## 15. Debugging with Logs

In [ ]:
import logging
from runtime_agents.shared.logger import get_logger

# Enable debug logging
logger = get_logger(__name__)
logger.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(levelname)s] %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

# Now run something and see detailed logs
orch = factory.create_orchestrator(llm, tools, "", "template_based")
results, final = await orch.run("Quick test.")
print(f"\nCompleted: {len(results)} agents")

---
## Summary

**Core Components:**
1. **LLM Client** - Talk to OpenAI
2. **Tools** - Agent capabilities
3. **Agent Templates** - Define roles
4. **Agent Instances** - Spawn and run
5. **Orchestrator** - Coordinate agents
6. **Session** - Persist state
7. **Factory** - Switch architectures

**6 Architectures:**
- Template-Based (predefined roles)
- LLM-Generated (dynamic creation)
- Compositional (reusable components)
- Meta-Agent (single adaptive)
- Hierarchical (tree structure)
- Evolutionary (self-improving)

**Key Patterns:**
- Tool scoping (agents get only needed tools)
- Context passing (agents build on each other)
- Performance tracking (compare architectures)

---

**Next:** Try modifying the examples, create your own agents/tools, explore different architectures!